# Scheme Utilisation — All Months Summary

**Objective:** Build a month-wise slab summary for 12 ML and 18 ML products across all available raw files.

**Output:** One Excel file (`scheme_utilisation_all_months.xlsx`) with two sheets:
- `12 ml` — one row per month, Slab 1 & Slab 2 metrics
- `18 ml` — one row per month, Slab 1–4 metrics

**Key rules:**
- Pack size identified via `SUb_Brand` column (contains `12 ML` or `18 ML`)
- Only `STOCKIEST DMS` distributor type is included
- Schemes with `SAMT` in the name are excluded
- Slab is assigned **per outlet per month** based on the outlet's **total quantity** that month — an outlet belongs to exactly one slab

## 1. Imports & Configuration

In [1]:
import re
import os
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd                      # used for fast file reading
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

# ---------------------------------------------------------------------------
# Paths — adjust RAW_FOLDER or OUTPUT if you move files
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # scheme_utilisation folder

RAW_FOLDER = PROJECT_ROOT / "Raw files"
OUTPUT     = PROJECT_ROOT / "scheme_utilisation_all_months.xlsx"

print(f"Raw files folder : {RAW_FOLDER}")
print(f"Output file      : {OUTPUT}")

Raw files folder : c:\Users\abqua\Desktop\HRI app\final_presentatiobn\scheme_utilisation\Raw files
Output file      : c:\Users\abqua\Desktop\HRI app\final_presentatiobn\scheme_utilisation\scheme_utilisation_all_months.xlsx


## 2. Helper Functions

In [2]:
# Month name → number mapping (handles both 'july' and 'jul' etc.)
MONTH_MAP = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5,
    "june": 6, "jun": 6, "july": 7, "jul": 7,
    "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

def parse_month_year(filename: str) -> datetime | None:
    """Extract a sortable datetime from a filename like 'Apr 24 Scheme Utilization.xlsx'."""
    name = filename.lower()
    match = re.match(r"([a-z]+)\s+(\d{2})\s+scheme", name)
    if match:
        mon = MONTH_MAP.get(match.group(1))
        yr  = 2000 + int(match.group(2))
        if mon:
            return datetime(yr, mon, 1)
    return None


def norm(value) -> str:
    """Return a safe stripped string — converts None and numbers to str."""
    return "" if value is None else str(value).strip()


def number(value) -> float:
    """Convert blank/None cells to 0.0, otherwise cast to float."""
    if value is None or value == "":
        return 0.0
    return float(value)


def get_pack(sub_brand: str) -> str | None:
    """Return '12 ml' or '18 ml' based on SUb_Brand text, or None if neither."""
    s = sub_brand.upper()
    if "12 ML" in s:
        return "12 ml"
    if "18 ML" in s:
        return "18 ml"
    return None


def slab_12ml(total_outlet_qty: float) -> str | None:
    """Assign slab for a 12 ML outlet based on its total monthly quantity."""
    if 8 <= total_outlet_qty < 144:
        return "Slab 1"
    if total_outlet_qty >= 144:
        return "Slab 2"
    return None  # below minimum — excluded


def slab_18ml(total_outlet_qty: float) -> str | None:
    """Assign slab for an 18 ML outlet based on its total monthly quantity."""
    if 8  <= total_outlet_qty < 32:  return "Slab 1"
    if 32 <= total_outlet_qty < 576: return "Slab 2"
    if 576 <= total_outlet_qty < 960: return "Slab 3"
    if total_outlet_qty >= 960:       return "Slab 4"
    return None  # below minimum — excluded


def empty_slab_summary() -> dict:
    """Accumulator dict for one slab — reset for each month."""
    return {
        "qty": 0.0,
        "weighted_discount_sum": 0.0,  # sum of (discount% * qty) for weighted avg
        "outlets": set(),
        "scheme_discount": 0.0,
        "distributor_sale_value": 0.0,
        "sale_value": 0.0,
    }

print("Helper functions defined.")

Helper functions defined.


## 3. Discover & Validate Raw Files

In [3]:
# Collect all .xlsx files, skipping temp files (prefixed with ~$)
all_files = [
    f for f in os.listdir(RAW_FOLDER)
    if f.endswith(".xlsx") and not f.startswith("~")
]

# Parse dates and sort chronologically
dated_files = []
unparsed   = []
for f in all_files:
    dt = parse_month_year(f)
    if dt:
        dated_files.append((dt, f))
    else:
        unparsed.append(f)

dated_files.sort(key=lambda x: x[0])

print(f"Files found      : {len(all_files)}")
print(f"Files parsed     : {len(dated_files)}")
if unparsed:
    print(f"Could not parse  : {unparsed}")

print("\nChronological order:")
for dt, f in dated_files:
    print(f"  {dt.strftime('%b %Y')}  →  {f}")

Files found      : 25
Files parsed     : 25

Chronological order:
  Apr 2024  →  Apr 24 Scheme Utilization.xlsx
  May 2024  →  May 24 Scheme Utilization.xlsx
  Jun 2024  →  June 24 Scheme Utilization.xlsx
  Jul 2024  →  July 24 Scheme Utilization.xlsx
  Aug 2024  →  Aug 24 Scheme Utilization.xlsx
  Sep 2024  →  Sep 24 Scheme Utilization.xlsx
  Oct 2024  →  Oct 24  Scheme Utilization.xlsx
  Nov 2024  →  Nov 24 Scheme Utilization.xlsx
  Dec 2024  →  Dec 24 Scheme Utilization.xlsx
  Jan 2025  →  Jan 25 Scheme Utilization.xlsx
  Feb 2025  →  Feb 25 Scheme Utilization.xlsx
  Mar 2025  →  Mar 25 Scheme Utilization.xlsx
  Apr 2025  →  Apr 25 Scheme Utilization.xlsx
  May 2025  →  may 25 Scheme Utilization.xlsx
  Jun 2025  →  June 25 Scheme Utilization.xlsx
  Jul 2025  →  July 25 Scheme Utilization.xlsx
  Aug 2025  →  Aug 25 Scheme Utilization.xlsx
  Sep 2025  →  Sep 25 Scheme Utilization.xlsx
  Oct 2025  →  Oct 25  Scheme Utilization.xlsx
  Nov 2025  →  Nov 25 Scheme Utilization.xlsx
  Dec 20

In [4]:
# ---------------------------------------------------------------------------
# Column consistency check — all files must have the same columns
# ---------------------------------------------------------------------------
base_cols = None
mismatches = []

for _, fname in dated_files:
    wb = openpyxl.load_workbook(RAW_FOLDER / fname, read_only=True, data_only=True)
    ws = wb.active
    cols = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
    wb.close()

    if base_cols is None:
        base_cols = cols
    elif cols != base_cols:
        extra   = set(cols) - set(base_cols)
        missing = set(base_cols) - set(cols)
        mismatches.append({"file": fname, "extra": extra, "missing": missing})

if mismatches:
    print("COLUMN MISMATCHES FOUND — check these files before proceeding:")
    for m in mismatches:
        print(f"  {m['file']}")
        print(f"    Extra columns  : {m['extra']}")
        print(f"    Missing columns: {m['missing']}")
else:
    print(f"All {len(dated_files)} files have identical columns. OK")

# Build column index from the first file (same for all)
col_idx = {name: i for i, name in enumerate(base_cols)}
print(f"\nTotal columns : {len(base_cols)}")

# Verify all required columns exist
REQUIRED = [
    "SUb_Brand", "Distributor_Type", "Outlet_Id", "Distributor code",
    "Distributor_id", "Bill No", "Invoice Date", "skunitid", "skucode",
    "Batch_Id", "Batch No", "Scheme Name", "Scheme_Reason",
    "%_Scheme", "Scheme_discount", "Invoice qty. pieces",
    "Distributor_Sale_Value", "Sale_Value",
]
missing_req = [c for c in REQUIRED if c not in col_idx]
if missing_req:
    raise ValueError(f"Required columns missing from files: {missing_req}")
print(f"All required columns present. OK")

All 25 files have identical columns. OK

Total columns : 47
All required columns present. OK


## 4. Process All Months

For each file (= one month):
1. Filter to `STOCKIEST DMS` only
2. Filter to `12 ML` or `18 ML` via `SUb_Brand`
3. Exclude schemes with `SAMT` in the name
4. Deduplicate invoice lines (multiple scheme rows can map to one invoice line)
5. Sum outlet-level total quantity, then assign each outlet to one slab
6. Accumulate 6 metrics per slab

In [5]:
# Only load the columns we actually need — keeps memory low for large files
COLS_NEEDED = [
    "SUb_Brand", "Distributor_Type", "Outlet_Id",
    "Distributor code", "Distributor_id", "Bill No",
    "Invoice Date", "skunitid", "skucode", "Batch_Id", "Batch No",
    "Scheme Name", "Scheme_Reason", "%_Scheme",
    "Invoice qty. pieces", "Scheme_discount",
    "Distributor_Sale_Value", "Sale_Value",
]

# These columns together uniquely identify one invoice line.
# Multiple scheme rows (Non-Claimable + Claimable) share the same key —
# qty and sale values must only be taken ONCE per key.
LINE_KEY_COLS = [
    "Distributor code", "Distributor_id", "Outlet_Id", "Bill No",
    "Invoice Date", "skunitid", "skucode", "Batch_Id", "Batch No",
]

month_results = {}   # {month_label: {pack: {slab: summary_dict}}}
month_order   = []   # [(datetime, month_label), ...]

for dt, fname in dated_files:
    month_label = dt.strftime("%b %Y")
    month_order.append((dt, month_label))
    print(f"Processing {month_label} ...", end="  ")

    # ------------------------------------------------------------------
    # Step 1 — Read only needed columns via pandas (fast)
    # ------------------------------------------------------------------
    df = pd.read_excel(RAW_FOLDER / fname, usecols=COLS_NEEDED)

    # ------------------------------------------------------------------
    # Step 2 — Apply base filters
    # ------------------------------------------------------------------
    # Keep STOCKIEST DMS only
    df = df[df["Distributor_Type"].str.strip() == "STOCKIEST DMS"]

    # Identify pack from SUb_Brand — add a "pack" column
    sb_upper = df["SUb_Brand"].str.upper().fillna("")
    df = df.copy()
    df["pack"] = None
    df.loc[sb_upper.str.contains("12 ML"), "pack"] = "12 ml"
    df.loc[sb_upper.str.contains("18 ML"), "pack"] = "18 ml"
    df = df[df["pack"].notna()]   # drop rows that are neither 12 ML nor 18 ML

    # Exclude SAMT schemes
    df = df[~df["Scheme Name"].str.upper().str.contains("SAMT", na=False)]

    # ------------------------------------------------------------------
    # Step 3 — Deduplication
    # Qty / Sale Values: take ONCE per invoice line (first occurrence)
    # Scheme Discount ₹: sum across ALL scheme rows for the same line (additive)
    # ------------------------------------------------------------------
    df["_line_key"] = df[LINE_KEY_COLS].astype(str).agg("|".join, axis=1)

    # First-occurrence values — qty and money values taken once per line
    first_occ = (
        df.drop_duplicates(subset="_line_key", keep="first")
        [["_line_key", "pack", "Outlet_Id",
          "Invoice qty. pieces", "Distributor_Sale_Value", "Sale_Value"]]
        .copy()
    )

    # Discount % — sum per line (Non-Claimable + Claimable both contribute)
    disc_pct = (
        df.groupby("_line_key")["%_Scheme"]
        .sum()
        .reset_index()
        .rename(columns={"%_Scheme": "total_discount_pct"})
    )

    # Scheme discount ₹ — additive across all scheme rows for the same line
    scheme_disc = (
        df.groupby("_line_key")["Scheme_discount"]
        .sum()
        .reset_index()
    )

    # Merge into one clean row per invoice line
    lines = (
        first_occ
        .merge(disc_pct,    on="_line_key")
        .merge(scheme_disc, on="_line_key")
    )

    # ------------------------------------------------------------------
    # Step 4 — Outlet-level total quantity (sum across all their lines)
    # Each outlet gets assigned to ONE slab based on this monthly total
    # ------------------------------------------------------------------
    outlet_total_qty = (
        lines.groupby(["pack", "Outlet_Id"])["Invoice qty. pieces"]
        .sum()
        .reset_index()
        .rename(columns={"Invoice qty. pieces": "outlet_total_qty"})
    )
    lines = lines.merge(outlet_total_qty, on=["pack", "Outlet_Id"])

    # ------------------------------------------------------------------
    # Step 5 — Assign ONE slab per outlet based on its total monthly qty
    # ------------------------------------------------------------------
    def assign_slab(row):
        qty = row["outlet_total_qty"]
        if row["pack"] == "12 ml":
            if 8   <= qty < 144: return "Slab 1"
            if qty >= 144:       return "Slab 2"
        else:
            if 8   <= qty < 32:  return "Slab 1"
            if 32  <= qty < 576: return "Slab 2"
            if 576 <= qty < 960: return "Slab 3"
            if qty >= 960:       return "Slab 4"
        return None   # below minimum threshold — excluded

    lines["slab"] = lines.apply(assign_slab, axis=1)
    lines = lines[lines["slab"].notna()]

    # ------------------------------------------------------------------
    # Step 6 — Accumulate 6 metrics per pack × slab
    # ------------------------------------------------------------------
    summaries = {
        "12 ml": defaultdict(empty_slab_summary),
        "18 ml": defaultdict(empty_slab_summary),
    }

    for _, row in lines.iterrows():
        target = summaries[row["pack"]][row["slab"]]
        qty = float(row["Invoice qty. pieces"])
        target["qty"]                    += qty
        target["weighted_discount_sum"]  += float(row["total_discount_pct"]) * qty
        target["outlets"].add(row["Outlet_Id"])
        target["scheme_discount"]        += float(row["Scheme_discount"])
        target["distributor_sale_value"] += float(row["Distributor_Sale_Value"])
        target["sale_value"]             += float(row["Sale_Value"])

    month_results[month_label] = summaries
    print(f"{len(lines):,} invoice lines | done")

print("\nAll months processed.")

Processing Apr 2024 ...  46,647 invoice lines | done
Processing May 2024 ...  50,544 invoice lines | done
Processing Jun 2024 ...  48,352 invoice lines | done
Processing Jul 2024 ...  55,599 invoice lines | done
Processing Aug 2024 ...  52,425 invoice lines | done
Processing Sep 2024 ...  48,147 invoice lines | done
Processing Oct 2024 ...  48,484 invoice lines | done
Processing Nov 2024 ...  43,457 invoice lines | done
Processing Dec 2024 ...  47,639 invoice lines | done
Processing Jan 2025 ...  58,186 invoice lines | done
Processing Feb 2025 ...  52,766 invoice lines | done
Processing Mar 2025 ...  58,678 invoice lines | done
Processing Apr 2025 ...  54,668 invoice lines | done
Processing May 2025 ...  53,217 invoice lines | done
Processing Jun 2025 ...  47,233 invoice lines | done
Processing Jul 2025 ...  50,327 invoice lines | done
Processing Aug 2025 ...  39,699 invoice lines | done
Processing Sep 2025 ...  43,505 invoice lines | done
Processing Oct 2025 ...  39,816 invoice lines 

## 5. Write Output Excel

In [6]:
PACK_SLABS = {
    "12 ml": ["Slab 1", "Slab 2"],
    "18 ml": ["Slab 1", "Slab 2", "Slab 3", "Slab 4"],
}

def write_sheet(wb: Workbook, sheet_name: str, slabs: list, month_order: list, month_results: dict):
    """Write one output sheet — one row per month, six metrics per slab."""
    ws = wb.create_sheet(sheet_name)

    # Build header row
    headers = ["Month"]
    for slab in slabs:
        headers.extend([
            f"{slab} Total Discount %",
            f"{slab} Total Quantity",
            f"{slab} Unique Store Count",
            f"{slab} Scheme Discount",
            f"{slab} Distributor Sale Value",
            f"{slab} Sale Value",
        ])
    ws.append(headers)

    # One data row per month in chronological order
    for _, month_label in month_order:
        summaries = month_results.get(month_label, {})
        pack_data = summaries.get(sheet_name, {})

        row = [month_label]
        for slab in slabs:
            data = pack_data.get(slab, empty_slab_summary())
            weighted_discount = data["weighted_discount_sum"] / data["qty"] if data["qty"] else 0
            row.extend([
                round(weighted_discount, 4),
                int(data["qty"]),
                len(data["outlets"]),
                round(data["scheme_discount"], 2),
                round(data["distributor_sale_value"], 2),
                round(data["sale_value"], 2),
            ])
        ws.append(row)

    # ---- Formatting ----
    # Bold + centered headers with wrap
    for cell in ws[1]:
        cell.font      = Font(bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 36

    # Center-align all data rows
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(horizontal="center", vertical="center")

    # Column widths
    ws.column_dimensions["A"].width = 12  # Month column
    for col_idx in range(2, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = 22

    # Number formats for data rows (groups of 6 per slab)
    for start_col in range(2, ws.max_column + 1, 6):
        for row_idx in range(2, ws.max_row + 1):
            ws.cell(row_idx, start_col).number_format     = "0.00"       # Discount %
            ws.cell(row_idx, start_col + 1).number_format = "#,##0"      # Quantity
            ws.cell(row_idx, start_col + 2).number_format = "#,##0"      # Store count
            ws.cell(row_idx, start_col + 3).number_format = "#,##0.00"   # Scheme discount ₹
            ws.cell(row_idx, start_col + 4).number_format = "#,##0.00"   # Distributor sale value ₹
            ws.cell(row_idx, start_col + 5).number_format = "#,##0.00"   # Sale value ₹

    print(f"Sheet '{sheet_name}' written — {ws.max_row - 1} month rows, {ws.max_column} columns")


# Build the workbook
output_wb = Workbook()
output_wb.remove(output_wb.active)  # remove default blank sheet

write_sheet(output_wb, "12 ml", PACK_SLABS["12 ml"], month_order, month_results)
write_sheet(output_wb, "18 ml", PACK_SLABS["18 ml"], month_order, month_results)

output_wb.save(OUTPUT)
print(f"\nSaved: {OUTPUT}")

Sheet '12 ml' written — 25 month rows, 13 columns
Sheet '18 ml' written — 25 month rows, 25 columns

Saved: c:\Users\abqua\Desktop\HRI app\final_presentatiobn\scheme_utilisation\scheme_utilisation_all_months.xlsx


## 6. Quick Sanity Check

In [7]:
# Re-open saved file and print a readable summary table
import pandas as pd

for sheet in ["12 ml", "18 ml"]:
    df = pd.read_excel(OUTPUT, sheet_name=sheet)
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet}  |  {df.shape[0]} months  |  {df.shape[1]} columns")
    print('='*60)

    # Show Quantity and Unique Store Count columns for a quick overview
    qty_cols  = [c for c in df.columns if "Quantity" in c]
    store_cols = [c for c in df.columns if "Store" in c]
    print(df[["Month"] + qty_cols + store_cols].to_string(index=False))


Sheet: 12 ml  |  25 months  |  13 columns
   Month  Slab 1 Total Quantity  Slab 2 Total Quantity  Slab 1 Unique Store Count  Slab 2 Unique Store Count
Apr 2024                 140979                  89783                       7837                        194
May 2024                 137253                 171501                       7346                        331
Jun 2024                 122087                 160401                       7022                        299
Jul 2024                 184960                  98363                      10091                        255
Aug 2024                 167178                 180171                       9137                        354
Sep 2024                 128995                 189737                       6450                        339
Oct 2024                 129120                 161994                       6261                        426
Nov 2024                 101287                 172313                       5018    